# PEAKS per-run QC metrics

One row per run, one number per metric, from PEAKS 13.x CLI output (DIA db
search), for tracking performance over time. All logic is in `peaks_run_qc.py`;
this notebook only calls it. Point `ROOT` at one PEAKS output folder or a parent
of many - every `*.dia_db.precursor.csv` below it is one run.

Reading notes:
- **Signal** (`total_ms2_area`, `total_ms1_area`) is summed precursor area, not a raw TIC.
  It is only comparable between runs searched **separately** - a multi-run PEAKS search
  normalises across runs and flattens exactly this number.
- `total_ms1_area` covers only the precursors that have an MS1 feature; track
  `frac_with_ms1` beside it.
- **FWHM, asymmetry, MS2 mass error** come from the PEAKS project database (the `<run>/`
  folder next to `exports/`), not the exports.
- **No filtering:** every metric uses all reported precursors; FWHM and asymmetry use
  every peak (one per PEAKS feature). FWHM is read at half the apex height. PEAKS stores
  each chromatogram only over its own integration window, and ~25% of peaks do not fall
  to half height inside it; there the window edge is taken as the crossing, so those
  widths are slightly low - stable between runs, so fine for trending. A peak whose apex
  sits on the window edge has no asymmetry (~4%).
- **Mass error** is reported as accuracy (median) and precision (MAD, unscaled). 11% of
  the MS1 values are exactly 0.0 (PEAKS writes that for some unmeasured errors); they are
  kept, which puts the MS1 median at 0 and narrows the MS1 MAD.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.dont_write_bytecode = True  # no __pycache__ next to the data
sys.path.insert(0, str(Path.cwd()))  # peaks_run_qc.py sits next to this notebook
import peaks_run_qc as qc

ROOT = Path(r"Z:\PEAKS_CLI")
pd.set_option("display.max_colwidth", 120)

In [ ]:
runs = qc.find_runs(ROOT)
runs[["search", "run"]]

In [ ]:
table = qc.collect(ROOT)
table.to_csv(ROOT / "peaks_run_qc.csv", index=False)
table.set_index(["search", "run"]).T

Metric definitions:

In [ ]:
qc.describe()

Distributions behind the numbers, one panel per run (saved to `qc_figures/`):

In [ ]:
for _, r in runs.iterrows():
    qc.plot_run_distributions(r, out_dir=ROOT / "qc_figures")

Trend across runs (only meaningful once there is more than one):

In [ ]:
import matplotlib.pyplot as plt

TREND = [
    "n_precursors",
    "n_protein_groups",
    "total_ms2_area",
    "asymmetry_geomean",
    "base_width_median_s",
    "ms1_ppm_sigma_iqr",
]
if len(table) > 1:
    t = table.sort_values("run")  # run names start with the acquisition date
    fig, axes = plt.subplots(len(TREND), 1, figsize=(8, 2.2 * len(TREND)), sharex=True)
    for ax, k in zip(axes, TREND):
        ax.plot(range(len(t)), t[k], "o-", color="#FA7A61", ms=4)
        ax.set_ylabel(k, fontsize=9)
    axes[-1].set_xticks(range(len(t)), t["run"], rotation=90, fontsize=7)
    fig.tight_layout()
else:
    print("one run - nothing to trend yet")